## L_03_3. Learning on graphs - an example

The aim of the notebook is to provide an example of a graph neural network. A numerical experiment, relying on GCN (see https://tkipf.github.io/graph-convolutional-networks/), is conducted for the Cora citation dataset.
See https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html#learning-methods-on-graphs 

### Modification for the assignment
The training pipeline was improved by normalizing the Cora node features, widening the hidden representation, fixing the random seed, and keeping the model checkpoint with the best validation accuracy.
This keeps the notebook close to the original GCN example while improving the final test result.

In [5]:
# Load data
import random
import numpy as np
import torch
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
print(len(dataset))
data = dataset[0]
print(data)

1
Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Processing...
Done!


In [6]:
# Build an improved GCN
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 32)
        self.conv2 = GCNConv(32, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)

        return x

In [7]:
# Train and keep the best validation checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN().to(device)
data = dataset[0].to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

best_val_acc = 0.0
best_test_acc = 0.0
best_epoch = 0
best_state = None

for epoch in range(1, 301):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    pred = model(data).argmax(dim=1)
    val_correct = (pred[data.val_mask] == data.y[data.val_mask]).sum()
    val_acc = int(val_correct) / int(data.val_mask.sum())
    test_correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    test_acc = int(test_correct) / int(data.test_mask.sum())

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
        best_epoch = epoch
        best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}

print(f'Best epoch: {best_epoch}')
print(f'Best validation accuracy: {best_val_acc:.4f}')

Best epoch: 210
Best validation accuracy: 0.8040


In [8]:
# Evaluate on the test set using the best validation checkpoint
model.load_state_dict(best_state)
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')
print(f'Best validation-selected test accuracy: {best_test_acc:.4f}')

Accuracy: 0.8120
Best validation-selected test accuracy: 0.8120
